[⬅ Back to the examples](https://anywidgetinstruments.github.io/anywidget-instruments-industrial/try/)

# Virtual instrument bench

Three instruments on one screen, as on a lab bench: a **function generator**
feeds an **oscilloscope** and a **multimeter**. Each front panel is made of
widgets: knobs and rotary selectors to set, an illuminated push button to
switch the output on, a screen and a display to read. **Run all the cells**,
then press **OUTPUT** and turn the knobs.

**References:** SI prefixes (engineering units). These standards inspired
the widgets; the library does not claim conformity with them ([Standards and
references](https://anywidgetinstruments.github.io/anywidget-instruments-industrial/standards/)).

> **The code is hidden.** To see a cell's code, click the `⋯` bar above its result; to see all of it, choose **View ▸ Expand All Code**.

In [ ]:
# JupyterLite: install the package (bundled with this site) in the browser kernel
%pip install -q anywidget-instruments-industrial

In [ ]:
import anywidget_instruments_industrial as ai

ai.theme_switch()  # light / system / dark theme (STYLE-008)

In [ ]:
import numpy as np

# function generator
wave = ai.SelectorSwitch("SIN", positions=["SIN", "SQR", "TRI", "SAW"], label="Waveform")
freq = ai.Knob(1000, min=10, max=5000, step=10, unit="Hz", label="Frequency", size=(120, 120))
amp = ai.Knob(1.0, min=0, max=5, step=0.1, unit="V", label="Amplitude", size=(120, 120))
offset = ai.Knob(0.0, min=-2.5, max=2.5, step=0.1, unit="V", label="Offset", size=(120, 120))
output = ai.PushButton(
    text="OUTPUT",
    mechanical_action="switch_when_pressed",
    lamp=False,
    lamp_color="green",
    label="Output",
)
# oscilloscope
tdiv = ai.SelectorSwitch("0.5 ms", positions=["0.1 ms", "0.5 ms", "1 ms", "5 ms"], label="Time/div")
vdiv = ai.SelectorSwitch("1 V", positions=["0.5 V", "1 V", "2 V"], label="Volts/div")
screen = ai.WaveformChart(
    history=500,
    update_mode="scope",
    x_unit="s",
    unit="V",
    traces=[{"name": "CH1"}],
    show_legend=False,
    label="Oscilloscope screen",
    size=(520, 240),
)
# multimeter
dmm_fn = ai.SelectorSwitch("VAC", positions=["VDC", "VAC", "Hz"], label="Function")
reading = ai.SevenSegment(0, digits=7, decimals=3, label="Multimeter reading", size=(230, 70))
overload = ai.LED(False, on_color="#dc2626", label="OVL", size=(40, 40))

CYCLE = {  # one period sampled at phase x in [0, 1)
    "SIN": lambda x: np.sin(2 * np.pi * x),
    "SQR": lambda x: np.where(x < 0.5, 1.0, -1.0),
    "TRI": lambda x: 1 - 4 * np.abs(x - 0.5),
    "SAW": lambda x: 2 * x - 1,
}
RMS = {"SIN": 1 / np.sqrt(2), "SQR": 1.0, "TRI": 1 / np.sqrt(3), "SAW": 1 / np.sqrt(3)}


def update(_change=None):
    on = bool(output.value)
    output.lamp = on
    a = amp.value if on else 0.0
    dc = offset.value if on else 0.0
    # oscilloscope: 10 divisions, 8 vertical divisions, triggered at phase 0
    t_div = {"0.1 ms": 1e-4, "0.5 ms": 5e-4, "1 ms": 1e-3, "5 ms": 5e-3}[tdiv.value]
    v_div = {"0.5 V": 0.5, "1 V": 1.0, "2 V": 2.0}[vdiv.value]
    t = np.arange(screen.history) * (10 * t_div / screen.history)
    screen.dt = 10 * t_div / screen.history
    screen.y_min, screen.y_max = -4 * v_div, 4 * v_div
    screen.clear()
    screen.append(dc + a * CYCLE[wave.value]((t * freq.value) % 1.0))
    # multimeter
    value = {"VDC": dc, "VAC": a * RMS[wave.value], "Hz": freq.value if a > 0 else 0.0}[
        dmm_fn.value
    ]
    reading.decimals = 0 if dmm_fn.value == "Hz" else 3
    reading.value = value
    overload.value = bool(dmm_fn.value != "Hz" and abs(value) > 9.999)


for control in (wave, freq, amp, offset, output, tdiv, vdiv, dmm_fn):
    control.on_change(update)
update()

## Function generator

In [ ]:
ai.Panel([wave, freq, amp, offset, output], columns=5)

## Oscilloscope and multimeter

In [ ]:
ai.Panel([screen, ai.Panel([reading, overload, dmm_fn], columns=2), tdiv, vdiv], columns=2)